# CS781 Capstone Phase 5: Legal Statute Identification (LSI)
## Submission 03: Model 2 AdaLoRA (Adaptive SVD Budget Allocation)
### Architectural Profile: Model 2 Wide Mistral Decoder (136.0M Parameters)
* **Backbone:** 12 layers, d=896, H_q=14, H_kv=7, GQA 2:1 (Base weights 100% FROZEN)
* **AdaLoRA Dynamic SVD Allocation:** Prunes from initial rank $r_{\text{init}} = 24$ down to target rank $r_{\text{target}} = 12$ across `q_proj`, `k_proj`, `v_proj`, `o_proj`
* **SVD Parameterization:** $\Delta W = P \Lambda Q$ with Frobenius orthogonality regularization on triplets $(P, \Lambda, Q)$
* **Precision:** Adapter parameters kept strictly in FP32 for numerical stability
* **Training:** 5 epochs with AdamW (LR_adalora=2.5e-4, LR_head=1.0e-3) with Asymmetric Loss (ASL)
* **Hub Sync:** Live non-blocking upload of trainable adapter weights to `Phase_5/sub03_model2_adalora/`
* **Evaluation:** Split-half threshold calibration (dev_tune -> dev_eval), Macro-F1 primary metric


In [ ]:
# 1. Environment & Self-Contained Phase 5 Core Library (100% Standalone)
import os
import sys
import gc
import json
import time
import math
import random
import zipfile
import shutil
import base64
import zlib
import types
import threading
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Any, Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ==============================================================================
# SELF-CONTAINED PHASE 5 CORE MODULE INLINING (Zero external file dependencies)
# ==============================================================================
_PHASE5_CORE_B64 = '...'

try:
    _core_src = zlib.decompress(base64.b64decode(_PHASE5_CORE_B64)).decode("utf-8")
    # 1. Write phase5_common.py directly to working directory so any subprocess or script can access it
    with open("phase5_common.py", "w", encoding="utf-8") as _f:
        _f.write(_core_src)
    
    # 2. Execute directly into an in-memory module
    p5 = types.ModuleType("phase5_common")
    exec(_core_src, p5.__dict__)
    sys.modules["phase5_common"] = p5
    print("Self-contained phase5_common core initialized successfully (100% standalone).")
except Exception as _e:
    print(f"Warning during self-contained core unpacking: {_e}")
    import phase5_common as p5

# Secure dynamic token retrieval (never hardcoded)
HF_TOKEN = p5.get_hf_token()
if HF_TOKEN:
    print("Hugging Face Hub authentication token identified successfully.")
else:
    print("WARNING: HF_TOKEN not found. Remote Hub sync will be local only.")


In [ ]:
# 2. Compute Device Configuration & Reproducibility
device = p5.get_best_device()
print(f"Active Compute Device: {device}")

p5.seed_everything(42)
print("Deterministic random seeds initialized (Seed 42).")

if device.type == "cuda":
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False)
    print("PyTorch SDPA memory-efficient attention enabled.")


In [ ]:
# 3. Experiment Configuration & Hyperparameters
# =====================================================================
# ⚙️ USER RUN CONTROL PANEL
# =====================================================================
# 1. PROFILE: Choose run mode:
#      - "core"  : Full production training on 100% of data (Official Phase 5 Run)
#      - "smoke" : Rapid end-to-end verification (<2 min on ~16-32 samples)
#                  Verifies model loading, forward, backward, calibration & zip packaging
PROFILE = "core"  # Options: "core" | "smoke"

# 2. EPOCHS: Number of training epochs (1 for smoke test; 5 for core production)
EPOCHS = 1 if PROFILE == "smoke" else 5

# 3. HUB_SYNC_STEP_INTERVAL: Checkpoint sync frequency to HF Hub (in optimizer steps)
HUB_SYNC_STEP_INTERVAL = 200
PAD_TOKEN_ID = p5.PAD_TOKEN_ID
# =====================================================================

EXPERIMENT_NAME = "sub03_model2_adalora"
MODEL_KEY = "model2"
SUBFOLDER = "model2-wide-mistral-136m"
TARGET_METRIC = "Macro-F1 (Official Primary Metric)"

OUTPUT_DIR = os.path.abspath(f"./checkpoint_{EXPERIMENT_NAME}")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
PREDICTION_JSON = os.path.join(OUTPUT_DIR, "prediction.json")
METRICS_JSON = os.path.join(OUTPUT_DIR, "val_metrics.json")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "loss_plot.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training Hyperparameters
MICRO_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size = 16
LR_ADALORA = 2.5e-4
LR_HEAD = 1.0e-3
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
ORTH_REG_WEIGHT = 0.1

# AdaLoRA Hyperparameters
R_INIT = 24
R_TARGET = 12
LORA_ALPHA = 32.0

# Chunking
CHUNK_WINDOW = 512
CHUNK_OVERLAP = 64
CHUNK_STRIDE = 448
TRAIN_SAMPLE_K = 6

print(f"Configuration set for: {EXPERIMENT_NAME}")
print(f"Run Profile: {PROFILE.upper()} | Configured Epochs: {EPOCHS} | Sync Interval: {HUB_SYNC_STEP_INTERVAL} steps")
print(f"AdaLoRA: R_init={R_INIT} -> R_target={R_TARGET}, Alpha={LORA_ALPHA}, Orth Reg Weight={ORTH_REG_WEIGHT}")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
# 4. Ingest Phase 5 Dataset & Setup Tokenizer
print("Ingesting Phase 5 LSI dataset...")
train_records, dev_tune, dev_eval, test_records, statute2idx, idx2statute = p5.ingest_phase5_dataset(HF_TOKEN)

# Handle SMOKE test profile slicing
if PROFILE == "smoke":
    print("=" * 65)
    print(">>> [PROFILE: SMOKE] Slicing dataset for rapid end-to-end verification <<<")
    print(f">>> Configured EPOCHS: {EPOCHS} | Hub Sync Interval: {HUB_SYNC_STEP_INTERVAL} steps")
    print("=" * 65)
    train_records = train_records[:32]
    dev_tune = dev_tune[:16]
    dev_eval = dev_eval[:16]
    test_records = test_records[:16]
    if EPOCHS > 2:
        print(f"[Smoke Mode Notice] Capping EPOCHS from {EPOCHS} to 1 for quick execution.")
        EPOCHS = 1

NUM_LABELS = len(statute2idx)
print(f"Total Unique Statutes (|L|): {NUM_LABELS}")
print(f"Train documents: {len(train_records):,}")
print(f"Dev Tune documents: {len(dev_tune):,}")
print(f"Dev Eval documents: {len(dev_eval):,}")
print(f"Test documents: {len(test_records):,}")

# Load custom scratch-trained 32K legal tokenizer
tokenizer = p5.load_legal_tokenizer(p5.HF_TOKENIZER_REPO, token=HF_TOKEN)
p5.PAD_TOKEN_ID = tokenizer.pad_token_id
PAD_TOKEN_ID = tokenizer.pad_token_id
if tokenizer.eos_token_id is not None:
    p5.EOS_TOKEN_ID = tokenizer.eos_token_id
    EOS_TOKEN_ID = tokenizer.eos_token_id
print(f"Loaded Tokenizer: Vocab={len(tokenizer):,}, Pad={tokenizer.pad_token_id}, EOS={tokenizer.eos_token_id}")


In [ ]:
# 5. AdaLoRA Dynamic SVD Allocation Architecture
class AdaLoRALinear(nn.Module):
    """
    AdaLoRA (Adaptive Budget Allocation for PEFT - Zhang et al., 2023).
    Weight update parameterization via SVD:
        Delta W = P @ diag(Lambda * mask) @ Q
    Triplets (P, Lambda, Q) maintain Frobenius orthogonality regularization.
    Keep adapter parameters in FP32 for numerical stability.
    """
    def __init__(self, base_layer: nn.Linear, r_init: int = 24, r_target: int = 12, alpha: float = 32.0):
        super().__init__()
        self.base_layer = base_layer
        self.r_init = r_init
        self.r_target = r_target
        self.alpha = alpha
        self.scaling = alpha / r_init

        # Freeze base layer completely
        self.base_layer.weight.requires_grad = False
        if self.base_layer.bias is not None:
            self.base_layer.bias.requires_grad = False

        d_in = base_layer.in_features
        d_out = base_layer.out_features

        # FP32 adapter parameters for SVD stability
        self.lora_P = nn.Parameter(torch.empty(d_out, r_init, dtype=torch.float32))
        self.lora_Lambda = nn.Parameter(torch.zeros(r_init, dtype=torch.float32))
        self.lora_Q = nn.Parameter(torch.empty(r_init, d_in, dtype=torch.float32))

        # Binary active rank mask
        self.register_buffer("rank_mask", torch.ones(r_init, dtype=torch.float32))

        # Orthonormal initialization
        nn.init.orthogonal_(self.lora_P)
        nn.init.orthogonal_(self.lora_Q)
        nn.init.zeros_(self.lora_Lambda)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig = self.base_layer(x)
        # SVD incremental path in FP32
        x_fp32 = x.float()
        q_out = F.linear(x_fp32, self.lora_Q)
        scaled_q = q_out * (self.lora_Lambda * self.rank_mask)
        p_out = F.linear(scaled_q, self.lora_P) * self.scaling
        return orig + p_out.to(x.dtype)

    def get_orthogonality_loss(self) -> torch.Tensor:
        # P^T P - I
        ptp = torch.matmul(self.lora_P.t(), self.lora_P)
        eye = torch.eye(self.r_init, device=self.lora_P.device, dtype=self.lora_P.dtype)
        p_loss = torch.norm(ptp - eye, p="fro") ** 2

        # Q Q^T - I
        qqt = torch.matmul(self.lora_Q, self.lora_Q.t())
        q_loss = torch.norm(qqt - eye, p="fro") ** 2
        return p_loss + q_loss

    def get_importance_scores(self) -> torch.Tensor:
        if self.lora_Lambda.grad is None:
            return torch.abs(self.lora_Lambda.data)
        lambda_imp = torch.abs(self.lora_Lambda.data * self.lora_Lambda.grad.data)
        p_imp = torch.mean(torch.abs(self.lora_P.data * (self.lora_P.grad.data if self.lora_P.grad is not None else 0)), dim=0)
        q_imp = torch.mean(torch.abs(self.lora_Q.data * (self.lora_Q.grad.data if self.lora_Q.grad is not None else 0)), dim=1)
        return lambda_imp + p_imp + q_imp

def apply_adalora_to_mistral(backbone: nn.Module, r_init: int = 24, r_target: int = 12, alpha: float = 32.0):
    for p in backbone.parameters():
        p.requires_grad = False

    adalora_modules = []
    for name, module in backbone.named_modules():
        if hasattr(module, "q_proj") and isinstance(module.q_proj, nn.Linear):
            module.q_proj = AdaLoRALinear(module.q_proj, r_init=r_init, r_target=r_target, alpha=alpha)
            adalora_modules.append(module.q_proj)
        if hasattr(module, "k_proj") and isinstance(module.k_proj, nn.Linear):
            module.k_proj = AdaLoRALinear(module.k_proj, r_init=r_init, r_target=r_target, alpha=alpha)
            adalora_modules.append(module.k_proj)
        if hasattr(module, "v_proj") and isinstance(module.v_proj, nn.Linear):
            module.v_proj = AdaLoRALinear(module.v_proj, r_init=r_init, r_target=r_target, alpha=alpha)
            adalora_modules.append(module.v_proj)
        if hasattr(module, "o_proj") and isinstance(module.o_proj, nn.Linear):
            module.o_proj = AdaLoRALinear(module.o_proj, r_init=r_init, r_target=r_target, alpha=alpha)
            adalora_modules.append(module.o_proj)

    return adalora_modules

print(f"Loading Model 2 Wide Mistral backbone from {p5.HF_MASTER_REPO}/{SUBFOLDER}...")
backbone, config_dict = p5.load_strict_phase1_backbone(MODEL_KEY, hf_token=HF_TOKEN)

adalora_layers = apply_adalora_to_mistral(backbone, r_init=R_INIT, r_target=R_TARGET, alpha=LORA_ALPHA)
print(f"Successfully applied AdaLoRA to {len(adalora_layers)} projection layers.")

model = p5.Phase5LSIModel(
    backbone=backbone,
    hidden_size=config_dict["hidden_size"],
    num_labels=NUM_LABELS,
    use_label_correlation=False
)
raw_model = model
if torch.cuda.device_count() > 1:
    print(f"Enabling DataParallel across {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(raw_model)
else:
    model = raw_model

model = model.to(device)

total_params = sum(p.numel() for p in raw_model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Architecture Parameters: {total_params / 1e6:.2f}M")
print(f"Trainable Parameters (AdaLoRA + Head + CAML): {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Frozen Base Backbone Parameters: {total_params - trainable_params:,}")


In [ ]:
# 6. Document Dataset with Citation-Prioritized Chunk Sampling
class DocumentLSIDataset(Dataset):
    def __init__(self, records, tokenizer, is_train=False, k=TRAIN_SAMPLE_K):
        self.records = records
        self.tokenizer = tokenizer
        self.is_train = is_train
        self.k = k

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        tokens = self.tokenizer.encode(rec["text"], add_special_tokens=False) + [self.tokenizer.eos_token_id]
        input_ids, attn_mask = p5.chunk_document_tokens(tokens, window_size=CHUNK_WINDOW, stride=CHUNK_STRIDE)

        if self.is_train:
            input_ids, attn_mask = p5.sample_training_chunks(
                input_ids, attn_mask, rec["text"], self.tokenizer, k=self.k
            )

        label_vec = torch.zeros(NUM_LABELS, dtype=torch.float32)
        for lbl in rec.get("labels", []):
            if int(lbl) in statute2idx:
                label_vec[statute2idx[int(lbl)]] = 1.0

        return input_ids, attn_mask, label_vec, rec["id"]

def collate_doc_batch(batch):
    max_m = max(item[0].size(0) for item in batch)
    s_len = batch[0][0].size(1)
    b_size = len(batch)

    padded_ids = torch.full((b_size, max_m, s_len), p5.PAD_TOKEN_ID, dtype=torch.long)
    padded_masks = torch.zeros((b_size, max_m, s_len), dtype=torch.long)
    chunk_mask = torch.zeros((b_size, max_m), dtype=torch.float32)
    targets = torch.stack([item[2] for item in batch], dim=0)
    doc_ids = [item[3] for item in batch]

    for i, item in enumerate(batch):
        m_len = item[0].size(0)
        padded_ids[i, :m_len, :] = item[0]
        padded_masks[i, :m_len, :] = item[1]
        valid_chunks = (item[1].sum(dim=-1) > 0).float()
        chunk_mask[i, :m_len] = valid_chunks

    return padded_ids, padded_masks, chunk_mask, targets, doc_ids

train_dataset = DocumentLSIDataset(train_records, tokenizer, is_train=True, k=TRAIN_SAMPLE_K)
dev_tune_dataset = DocumentLSIDataset(dev_tune, tokenizer, is_train=False)
dev_eval_dataset = DocumentLSIDataset(dev_eval, tokenizer, is_train=False)
test_dataset = DocumentLSIDataset(test_records, tokenizer, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=True, collate_fn=collate_doc_batch)
dev_tune_loader = DataLoader(dev_tune_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=collate_doc_batch)
dev_eval_loader = DataLoader(dev_eval_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=collate_doc_batch)
test_loader = DataLoader(test_dataset, batch_size=MICRO_BATCH_SIZE, shuffle=False, collate_fn=collate_doc_batch)

print("DataLoaders initialized successfully with citation-prioritized chunking.")


In [ ]:
# 7. Pre-Flight Verification & Orthogonality Check
print("Running mandatory 1-batch pre-flight gradient verification...")
model.train()

test_batch = next(iter(train_loader))
b_ids, b_masks, b_cmask, b_targets, _ = test_batch

b_ids = b_ids.to(device)
b_masks = b_masks.to(device)
b_cmask = b_cmask.to(device)
b_targets = b_targets.to(device)

criterion = p5.AsymmetricLoss(gamma_neg=4.0, gamma_pos=0.0, clip_margin=0.05)

test_logits = model(b_ids, b_masks, chunk_mask=b_cmask)
asl_loss = criterion(test_logits, b_targets)

# Compute orthogonality regularization across AdaLoRA layers
orth_loss = sum(mod.get_orthogonality_loss() for mod in adalora_layers) / len(adalora_layers)
total_loss = asl_loss + ORTH_REG_WEIGHT * orth_loss

assert not torch.isnan(total_loss), "Pre-flight total loss produced NaN!"
total_loss.backward()

# MANDATORY ASSERTIONS:
adalora_P_grads = [mod.lora_P.grad.norm().item() for mod in adalora_layers if mod.lora_P.grad is not None]
assert any(g > 0 for g in adalora_P_grads), "All AdaLoRA P matrices received zero gradient!"

assert raw_model.classifier_head.label_weights.grad is not None and raw_model.classifier_head.label_weights.grad.norm().item() > 0, (
    "Zero gradient on classifier head weights!"
)
assert raw_model.chunk_attention.label_queries.grad is not None and raw_model.chunk_attention.label_queries.grad.norm().item() > 0, (
    "Zero gradient on CAML query parameters!"
)

# Base weights strictly frozen
for name, p in raw_model.named_parameters():
    if "base_layer" in name:
        assert p.grad is None or p.grad.norm().item() == 0, f"LEAKAGE: Base weight {name} received gradient!"

model.zero_grad()
print("Pre-flight gradient assertions PASSED successfully! SVD parameters verified.")


In [ ]:
# 8. Dynamic SVD Budget Allocation Routine & Optimizer Setup
total_training_steps = (len(train_loader) // GRADIENT_ACCUMULATION_STEPS) * EPOCHS
t_init = int(total_training_steps * 0.10)
t_final = int(total_training_steps * 0.85)

def update_and_allocate(step: int):
    """
    Cubic rank budget allocation: prunes least important triplets across layers
    from R_INIT (24) down to R_TARGET (12).
    """
    if step < t_init:
        current_target_r = R_INIT
    elif step > t_final:
        current_target_r = R_TARGET
    else:
        progress = float(step - t_init) / float(max(1, t_final - t_init))
        current_target_r = int(R_TARGET + (R_INIT - R_TARGET) * (1.0 - progress) ** 3)

    # Collect importance scores across all AdaLoRA layers
    all_scores = []
    for mod in adalora_layers:
        scores = mod.get_importance_scores()
        all_scores.append(scores)

    # Stack scores: [num_layers, R_INIT]
    stacked_scores = torch.stack(all_scores, dim=0)

    # Calculate global threshold for current_target_r budget
    k_keep = current_target_r * len(adalora_layers)
    flattened = stacked_scores.view(-1)
    k_keep = max(1, min(k_keep, len(flattened)))
    threshold = torch.topk(flattened, k_keep, largest=True).values[-1]

    # Update active binary rank masks
    for i, mod in enumerate(adalora_layers):
        mask = (all_scores[i] >= threshold).float()
        # Guarantee at least 1 rank remains active per layer
        if mask.sum() == 0:
            mask[all_scores[i].argmax()] = 1.0
        mod.rank_mask.copy_(mask)

no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight", "label_queries", "query", "layer_weights"]
optimizer_grouped_parameters = [
    {
        "params": [p for n, p in raw_model.named_parameters() if "lora_" in n and p.requires_grad and not any(nd in n for nd in no_decay)],
        "lr": LR_ADALORA,
        "weight_decay": WEIGHT_DECAY
    },
    {
        "params": [p for n, p in raw_model.named_parameters() if "lora_" in n and p.requires_grad and any(nd in n for nd in no_decay)],
        "lr": LR_ADALORA,
        "weight_decay": 0.0
    },
    {
        "params": [p for n, p in raw_model.named_parameters() if not n.startswith("backbone") and p.requires_grad and not any(nd in n for nd in no_decay)],
        "lr": LR_HEAD,
        "weight_decay": WEIGHT_DECAY
    },
    {
        "params": [p for n, p in raw_model.named_parameters() if not n.startswith("backbone") and p.requires_grad and any(nd in n for nd in no_decay)],
        "lr": LR_HEAD,
        "weight_decay": 0.0
    }
]

optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_training_steps, eta_min=1e-5)

hub_sync = p5.ResilientHubSyncManager(
    repo_id=p5.HF_MASTER_REPO,
    subfolder=f"Phase_5/{EXPERIMENT_NAME}",
    token=HF_TOKEN,
    is_peft=True
)

print(f"Total optimization steps: {total_training_steps:,} (Pruning window: {t_init}..{t_final})")
print(f"Optimization setup: AdaLoRA LR={LR_ADALORA}, Head LR={LR_HEAD}")


In [ ]:
# 9. AdaLoRA Training Execution Loop
scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))
global_step = 0
best_eval_f1 = -1.0
history = {"train_loss": [], "eval_macro_f1": []}

print(f"Starting AdaLoRA fine-tuning for {EPOCHS} epochs...")
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (b_ids, b_masks, b_cmask, b_targets, _) in enumerate(train_loader):
        b_ids = b_ids.to(device)
        b_masks = b_masks.to(device)
        b_cmask = b_cmask.to(device)
        b_targets = b_targets.to(device)

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16):
            logits = model(b_ids, b_masks, chunk_mask=b_cmask)
            asl_loss = criterion(logits, b_targets)
            orth_loss = sum(mod.get_orthogonality_loss() for mod in adalora_layers) / len(adalora_layers)
            total_loss = (asl_loss + ORTH_REG_WEIGHT * orth_loss) / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(total_loss).backward()
        running_loss += total_loss.item() * GRADIENT_ACCUMULATION_STEPS

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            global_step += 1

            # Update rank allocation budget
            if global_step % 10 == 0:
                update_and_allocate(global_step)

            if (global_step % HUB_SYNC_STEP_INTERVAL == 0) or (PROFILE == "smoke" and (step + 1) == len(train_loader) and epoch == EPOCHS):
                print(f"[HubSync] Overwriting latest/ adapter weights at step {global_step}...")
                hub_sync.save_and_sync_step(
                    step=global_step,
                    epoch=epoch,
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    local_dir=OUTPUT_DIR,
                    is_best=False
                )

    avg_train_loss = running_loss / len(train_loader)
    history["train_loss"].append(avg_train_loss)

    # Evaluate on dev_tune split to select optimal checkpoint (prevents selection leak on dev_eval)
    model.eval()
    tune_probs_list, tune_targets_list = [], []
    with torch.no_grad():
        for b_ids, b_masks, b_cmask, b_targets, _ in dev_tune_loader:
            b_ids = b_ids.to(device)
            b_masks = b_masks.to(device)
            b_cmask = b_cmask.to(device)
            with torch.amp.autocast(device_type=device.type):
                logits = model(b_ids, b_masks, chunk_mask=b_cmask)
            probs = torch.sigmoid(logits).cpu().numpy()
            tune_probs_list.append(probs)
            tune_targets_list.append(b_targets.numpy())

    epoch_tune_probs = np.vstack(tune_probs_list)
    epoch_tune_targets = np.vstack(tune_targets_list)
    temp_preds = (epoch_tune_probs >= 0.30).astype(np.int32)
    zero_mask = (temp_preds.sum(axis=1) == 0)
    temp_preds[zero_mask, epoch_tune_probs[zero_mask].argmax(axis=1)] = 1

    epoch_tune_f1 = p5.compute_lsi_metrics(epoch_tune_targets, temp_preds)["macro_f1"]
    history["eval_macro_f1"].append(epoch_tune_f1)

    is_best = epoch_tune_f1 > best_eval_f1
    if is_best:
        best_eval_f1 = epoch_tune_f1
        best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_model_state, os.path.join(OUTPUT_DIR, "best_adalora_weights.pt"))

    # Active rank budget statistics
    active_ranks = [int(mod.rank_mask.sum().item()) for mod in adalora_layers]
    avg_rank = sum(active_ranks) / len(active_ranks)
    print(f"Epoch [{epoch}/{EPOCHS}] | Step: {global_step} | Avg Active Rank: {avg_rank:.1f} | Train Loss: {avg_train_loss:.4f} | Dev Tune F1: {epoch_tune_f1:.4f} {'(New Peak!)' if is_best else ''}")

    # Push best checkpoint if improved
    hub_sync.save_and_sync_step(
        step=global_step,
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scheduler=scheduler,
        local_dir=OUTPUT_DIR,
        val_macro_f1=epoch_tune_f1,
        is_best=is_best
    )

train_time = time.time() - start_time
print(f"AdaLoRA training completed in {train_time / 3600:.2f} hours. Peak Macro-F1: {best_eval_f1:.4f}")


In [ ]:
# 10. Training Loss and Validation Trajectory Plot
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history["train_loss"], marker='o', color='teal', label="Train ASL + Orth Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{EXPERIMENT_NAME} - Loss Curve")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history["eval_macro_f1"], marker='s', color='darkorange', label="Dev Eval Macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Validation Metric Progression")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()
print(f"Saved performance curves to {LOSS_PLOT_PATH}")


In [ ]:
# 11. Split-Half Calibration on dev_tune & Unbiased Reporting on dev_eval
# Restore best checkpoint snapshot before calibration
if 'best_model_state' in locals() and best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("Restored in-memory snapshot of best checkpoint before calibration.")
elif os.path.exists(os.path.join(OUTPUT_DIR, "best_adalora_weights.pt")):
    model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_adalora_weights.pt"), map_location=device), strict=False)
    print("Loaded best AdaLoRA adapter weights for evaluation.")

model.eval()

def gather_predictions(loader):
    probs_list, targets_list, ids_list = [], [], []
    with torch.no_grad():
        for b_ids, b_masks, b_cmask, b_targets, b_doc_ids in loader:
            b_ids = b_ids.to(device)
            b_masks = b_masks.to(device)
            b_cmask = b_cmask.to(device)
            with torch.amp.autocast(device_type=device.type):
                logits = model(b_ids, b_masks, chunk_mask=b_cmask)
            probs = torch.sigmoid(logits).cpu().numpy()
            probs_list.append(probs)
            targets_list.append(b_targets.numpy())
            ids_list.extend(b_doc_ids)
    return np.vstack(probs_list), np.vstack(targets_list), ids_list

print("Gathering validation split predictions...")
tune_probs, tune_targets, tune_ids = gather_predictions(dev_tune_loader)
eval_probs, eval_targets, eval_ids = gather_predictions(dev_eval_loader)

# Calibrate on dev_tune
calibrator = p5.SplitHalfCalibrator(statute2idx, idx2statute)
train_label_support = np.zeros(NUM_LABELS, dtype=np.int32)
for rec in train_records:
    for lbl in rec["labels"]:
        if int(lbl) in statute2idx:
            train_label_support[statute2idx[int(lbl)]] += 1

calibrator.fit(tune_targets, tune_probs, train_support=train_label_support)

# Unbiased evaluation on dev_eval
eval_binary_preds = calibrator.predict(eval_probs)
unbiased_metrics = p5.compute_lsi_metrics(eval_targets, eval_binary_preds)

print("\n" + "=" * 60)
print("UNBIASED DEV_EVAL VALIDATION PERFORMANCE (SUB 03)")
print("=" * 60)
for k, v in unbiased_metrics.items():
    print(f"  {k:20s}: {v:.4f}")
print("=" * 60)

with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(unbiased_metrics, f, indent=2)
print(f"Saved unbiased validation metrics to {METRICS_JSON}")


In [ ]:
# 12. Test Set Inference & Strict Codabench Packaging
print("Generating test predictions on official test set...")
test_probs_list, test_ids_list = [], []

with torch.no_grad():
    for b_ids, b_masks, b_cmask, _, b_doc_ids in test_loader:
        b_ids = b_ids.to(device)
        b_masks = b_masks.to(device)
        b_cmask = b_cmask.to(device)
        with torch.amp.autocast(device_type=device.type):
            logits = model(b_ids, b_masks, chunk_mask=b_cmask)
        probs = torch.sigmoid(logits).cpu().numpy()
        test_probs_list.append(probs)
        test_ids_list.extend(b_doc_ids)

test_probs = np.vstack(test_probs_list)
print(f"Test probability matrix shape: {test_probs.shape}")

# Save full dev probabilities (tune + eval) and test probabilities for Sub 07 ensemble
dev_probs_combined = np.vstack([tune_probs, eval_probs])
np.save(os.path.join(OUTPUT_DIR, "dev_probs.npy"), dev_probs_combined)
np.save(os.path.join(OUTPUT_DIR, "sub03_test_probs.npy"), test_probs)

# Format JSON predictions
test_predictions = calibrator.format_predictions_json(test_probs, test_ids_list)

# Validate and package into submission.zip
p5.package_codabench_submission(
    predictions_list=test_predictions,
    output_zip_path=SUBMISSION_ZIP,
    expected_doc_ids=test_ids_list
)
print("Codabench submission archive verified and ready.")


In [ ]:
# 13. Atomically Push Sub 03 Artifacts to Master Hugging Face Hub
if HF_TOKEN:
    print(f"Uploading Sub 03 deliverables to {p5.HF_MASTER_REPO}/Phase_5/{EXPERIMENT_NAME}...")
    try:
        api = p5.HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=SUBMISSION_ZIP,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/submission.zip",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 03 Codabench submission.zip"
        )
        api.upload_file(
            path_or_fileobj=PREDICTION_JSON,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/prediction.json",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 03 prediction.json"
        )
        api.upload_file(
            path_or_fileobj=METRICS_JSON,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/val_metrics.json",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 03 val_metrics.json"
        )
        if os.path.exists(LOSS_PLOT_PATH):
            api.upload_file(
                path_or_fileobj=LOSS_PLOT_PATH,
                path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/loss_plot.png",
                repo_id=p5.HF_MASTER_REPO,
                commit_message="Add Sub 03 loss plot"
            )
        dev_prob_p = os.path.join(OUTPUT_DIR, "dev_probs.npy")
        if os.path.exists(dev_prob_p):
            api.upload_file(path_or_fileobj=dev_prob_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/dev_probs.npy", repo_id=p5.HF_MASTER_REPO, commit_message="Add dev_probs.npy")
        test_prob_p = os.path.join(OUTPUT_DIR, "sub03_test_probs.npy")
        if os.path.exists(test_prob_p):
            api.upload_file(path_or_fileobj=test_prob_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/sub03_test_probs.npy", repo_id=p5.HF_MASTER_REPO, commit_message="Add sub03_test_probs.npy")
        weights_p = os.path.join(OUTPUT_DIR, "best_model_weights.pt")
        if os.path.exists(weights_p):
            api.upload_file(path_or_fileobj=weights_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/best_model_weights.pt", repo_id=p5.HF_MASTER_REPO, commit_message="Add best model weights")
        hub_sync.join()
        print("Sub 03 artifacts uploaded to Hugging Face Hub successfully.")
    except Exception as e:
        print(f"Hub upload notice: {e}. Checkpoint remains safely stored locally in {OUTPUT_DIR}")
else:
    print("HF_TOKEN omitted: All artifacts safely stored locally.")
